# Figures for the paper (Version 6)

This notebook redraws all 13 figures of the revised manuscript from the tables written by `Analysis.ipynb` (`summary_with_CI.csv`, `data_vs_model_by_group.csv`, `point_estimates_by_seed.csv` in `Results/`). The plotting code is the same as in `Analysis.ipynb`; it is collected here so that the figures can be restyled without re-running the bootstrap. It replaces the earlier Excel-based `Plots.ipynb`.

Markers: ↓/↑ = significant decrease/increase relative to the original (or full) model, two-sided paired patient-level bootstrap, Holm-adjusted p < 0.05 within each library × metric.

In [ ]:
import os, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

# Folder with the outputs of Analysis.ipynb (summary_with_CI.csv, data_vs_model_by_group.csv)
BASE     = os.environ.get("MIDRC_RESULTS", os.path.join(os.getcwd(), "Results"))
PLOT_DIR = os.environ.get("MIDRC_PLOTS", os.path.join(os.getcwd(), "Plots"))
os.makedirs(PLOT_DIR, exist_ok=True)

summary   = pd.read_csv(os.path.join(BASE, "summary_with_CI.csv"))
groups_tbl = pd.read_csv(os.path.join(BASE, "data_vs_model_by_group.csv"))
SEEDS = sorted(pd.read_csv(os.path.join(BASE, "point_estimates_by_seed.csv"))["seed"].unique())
print(len(SEEDS), "seeds |", summary[["library", "method"]].drop_duplicates().shape[0], "configurations")

# configuration -> prediction file and stage, in the order used by the figures
METHODS = {
    "Fairlearn": [
        ("Original Model", "y_pred_Original.csv", ""),
        ("Correlation Remover", "y_pred_Corr.csv", "Pre-processing"),
        ("Exponentiated Gradient", "y_pred_expGrad.csv", "In-processing"),
        ("Grid Search", "y_pred_gridsearch.csv", "In-processing"),
        ("Threshold Optimizer", "y_pred_Threshold.csv", "Post-processing"),
        ("Correlation Remover → Exponentiated Gradient", "y_pred_Corr_ExpGrad.csv", "Sequential Algorithms"),
        ("Grid Search → Threshold Optimizer", "y_pred_Grid_Threshold.csv", "Sequential Algorithms"),
        ("Correlation Remover → Threshold Optimizer", "y_pred_Corr_Threshold.csv", "Sequential Algorithms"),
    ],
    "AIF360": [
        ("Original", "y_pred_Original.csv", ""),
        ("Reweighing", "y_pred_Reweighing.csv", "Preprocessing Algorithms"),
        ("Disparate Impact Remover", "y_pred_DisparateImpactRemover.csv", "Preprocessing Algorithms"),
        ("Learning Fair Representations", "y_pred_LFR.csv", "Preprocessing Algorithms"),
        ("Exponentiated Gradient Reduction", "y_pred_EGR.csv", "In-processing Algorithms"),
        ("Grid Search Reduction", "y_pred_GSR.csv", "In-processing Algorithms"),
        ("Equalized Odds Postprocessing", "y_pred_EOP.csv", "Postprocessing Algorithms"),
        ("Calibrated Equalized Odds Postprocessing", "y_pred_CEOP.csv", "Postprocessing Algorithms"),   # Version 6
        ("Reject Option Classification", "y_pred_ROC.csv", "Postprocessing Algorithms"),
        ("RW + EGR", "y_pred_Sequential_RW_EGR.csv", "Sequential Models"),
        ("DIR + GSR", "y_pred_Sequential_DIR_GSR.csv", "Sequential Models"),
        ("EGR + CEOP", "y_pred_Sequential_EGR_CEOP.csv", "Sequential Models"),
        ("RW + CEOP", "y_pred_Sequential_RW_CEOP.csv", "Sequential Models"),
    ],
    # Version 6 (Reviewer 1, comment 2): input ablations of the Original Model (written by
    # Fairlearn.ipynb; same architecture, folds and fold-specific PCA)
    "Ablation": [
        ("Full model", "y_pred_Original.csv", "image + age + sex + ethnicity"),
        ("Image + age + sex", "y_pred_Abl_ImageAgeSex.csv", "ethnicity removed"),
        ("Image + age", "y_pred_Abl_ImageAge.csv", "no sensitive attribute"),
        ("Image only", "y_pred_Abl_ImageOnly.csv", "no demographic input"),
        ("Demographics only", "y_pred_Abl_Demographics.csv", "no image input"),
    ],
}

# configurations present in the summary (replaces the bootstrap table used in Analysis.ipynb)
BOOT = set(map(tuple, summary[["library", "method"]].drop_duplicates().values))


## Figure 1: data bias versus model bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [2, 5]})
for ax, attr in zip(axes, ["Gender", "Ethnicity"]):
    t = groups_tbl[groups_tbl.attribute == attr]
    x = np.arange(len(t)); w = 0.38
    ax.bar(x - w / 2, 100 * t["COVID-19 base rate (data)"], w, color="#7b9cda", edgecolor="black", hatch="///", label="COVID-19 base rate in the data")
    ax.bar(x + w / 2, 100 * t["predicted COVID-19 rate (model)"], w, color="#f4845f", edgecolor="black", hatch="xxx", label="Predicted COVID-19 rate (Original Model)")
    for xi, n in zip(x, t["n"]):
        ax.text(xi, 2, f"n={n}", ha="center", fontsize=8, color="white", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(t["group"]); ax.set_ylim(0, 100); ax.set_title(attr)
    ax.set_ylabel("Rate (%)"); ax.grid(axis="y", linestyle="--", alpha=0.4)
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Data bias versus model bias", fontsize=13)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Data-vs-Model-Bias.png"), dpi=150, bbox_inches="tight"); plt.show()

## Figures 3–11: Fairlearn and AIF360 metrics with accuracy

In [ ]:
COLORS = ["#2ec4b6", "#f4845f", "#7b9cda", "#d966a0", "#8bc34a"]
HATCHES = ["///", "///", "xxx", "...", "///"]
import textwrap

def get(lib, name, col):
    r = summary[(summary.library == lib) & (summary.method == name) & (summary.metric == col)]
    return (r["mean"].iloc[0], r["ci_low"].iloc[0], r["ci_high"].iloc[0], r["p_holm"].iloc[0]) if len(r) else (np.nan,) * 4

# Version 6 (Reviewer 1, comment 4): direction-specific markers. ↓ = significant decrease, ↑ = significant
# increase relative to the original model (two-sided paired patient-level bootstrap, Holm-adjusted
# p < 0.05 within the library x metric family). For the disparity metrics plotted here ↓ is an improvement.
def mark(lib, name, col):
    r = summary[(summary.library == lib) & (summary.method == name) & (summary.metric == col)]
    return {"decrease": "↓", "increase": "↑"}.get(r["significant_change"].iloc[0], "") if len(r) else ""

def plot(lib, cols, labels, title, fname, ylabel="Metric value (%)"):
    items = [(n, st) for n, _, st in METHODS[lib] if not summary[(summary.library == lib) & (summary.method == n)].empty]
    names = [n for n, _ in items]; cats = [st for _, st in items]
    n, k = len(names), len(cols)
    bw = min(0.8 / k, 0.22); x = np.arange(n); off = (np.arange(k) - (k - 1) / 2) * bw
    fig, ax1 = plt.subplots(figsize=(max(12, 1.7 * n), 7.5)); ax2 = ax1.twinx()
    top = 0
    for j, (col, lab) in enumerate(zip(cols, labels)):
        v = np.array([get(lib, nm, col) for nm in names], float)
        m, lo, hi, p = v[:, 0] * 100, v[:, 1] * 100, v[:, 2] * 100, v[:, 3]
        ax1.bar(x + off[j], m, bw, color=COLORS[j % 5], hatch=HATCHES[j % 5], edgecolor="black", linewidth=0.6, label=lab, zorder=3)
        ax1.errorbar(x + off[j], m, yerr=np.clip(np.array([m - lo, hi - m], dtype=float), 0, None), fmt="none", ecolor="black", elinewidth=0.9, capsize=2.5, zorder=4)
        for xi, mi, hii, nm in zip(x + off[j], m, hi, names):
            ax1.text(xi, hii + 0.6, f"{mi:.1f}{mark(lib, nm, col)}", ha="center", va="bottom", fontsize=6.5, rotation=90)
        top = max(top, np.nanmax(hi))
    acc = np.array([get(lib, nm, "Accuracy") for nm in names], float) * 100
    ax2.errorbar(x, acc[:, 0], yerr=np.clip(np.array([acc[:, 0] - acc[:, 1], acc[:, 2] - acc[:, 0]], dtype=float), 0, None), color="black", marker="o", ms=6, lw=1.6, capsize=3, zorder=5)
    for xi, a in zip(x, acc[:, 0]):
        ax2.text(xi, a + 2, f"{a:.1f}%", ha="center", fontsize=8.5, fontweight="bold")
    ax1.set_ylim(0, max(10, np.ceil(top * 1.35 / 5) * 5)); ax2.set_ylim(0, 100)
    ax1.set_xlim(-0.5, n - 0.5); ax1.set_xticks(x); ax1.set_xticklabels([textwrap.fill(s, 16) for s in names], fontsize=8.5)
    ax1.set_ylabel(ylabel); ax2.set_ylabel("Accuracy (%)"); ax1.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    start = 0
    for i in range(1, n + 1):
        if i == n or cats[i] != cats[start]:
            if cats[start]:
                ax1.text((start + i - 1) / 2, -0.17, cats[start], transform=ax1.get_xaxis_transform(), ha="center", va="top", fontsize=9, style="italic", color="#444")
            if i < n:
                ax1.axvline(i - 0.5, color="#999", ls=":", lw=1)
            start = i
    handles = [mpatches.Patch(facecolor=COLORS[j % 5], hatch=HATCHES[j % 5], edgecolor="black", label=l) for j, l in enumerate(labels)]
    handles.append(plt.Line2D([0], [0], color="black", marker="o", label="Accuracy"))
    ax1.legend(handles=handles, loc="upper left", fontsize=8.5, framealpha=0.9)
    ax1.set_title(title + f"  (mean of {len(SEEDS)} seeds, 95% CI; ↓/↑ significant decrease/increase vs. original, Holm)", fontsize=11)
    fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, fname), dpi=150, bbox_inches="tight"); plt.show()

for attr, tag in [("Gender", "Gender"), ("Ethnicity", "Ethnicity"), ("Gender+Ethnicity", "Gender-Ethnicity")]:
    plot("Fairlearn", [f"{attr}|DPD"], ["Demographic Parity Difference (COVID-19)"], f"Fairlearn - {attr}",
         f"Fairlearn-{tag}-Demographic_Parity.png", "Demographic Parity (%)")
    plot("Fairlearn", [f"{attr}|EOD"], ["Equalized Odds Difference (COVID-19)"], f"Fairlearn - {attr}",
         f"Fairlearn-{tag}-Equalized_Odds.png", "Equalized Odds (%)")
    plot("AIF360", [f"{attr}|{m}" for m in ["SPD", "AOD", "EOpD", "Theil", "GEI"]],
         ["Statistical Parity Difference", "Average Odds Difference", "Equal Opportunity Difference",
          "Theil Index", "Generalized Entropy Index"], f"AIF360 - {attr}", f"AIF360-{tag}.png", "Metrics Values (%)")
print(sorted(os.listdir(PLOT_DIR)))

## Figures 2, 12, 13: ablations, subgroup sensitivity, fairness–utility trade-off

In [ ]:
SHORT = {"Original Model": "Original", "Correlation Remover": "CR", "Exponentiated Gradient": "EG", "Grid Search": "GS",
         "Threshold Optimizer": "TO", "Correlation Remover → Exponentiated Gradient": "CR→EG",
         "Grid Search → Threshold Optimizer": "GS→TO", "Correlation Remover → Threshold Optimizer": "CR→TO",
         "Original": "Original", "Reweighing": "RW", "Disparate Impact Remover": "DIR",
         "Learning Fair Representations": "LFR", "Exponentiated Gradient Reduction": "EGR",
         "Grid Search Reduction": "GSR", "Equalized Odds Postprocessing": "EOP",
         "Calibrated Equalized Odds Postprocessing": "CEOP", "Reject Option Classification": "ROC",
         "RW + EGR": "RW+EGR", "DIR + GSR": "DIR+GSR", "EGR + CEOP": "EGR+CEOP", "RW + CEOP": "RW+CEOP"}
STAGE_COLOR = {"": "#333333", "Pre-processing": "#2ec4b6", "Preprocessing Algorithms": "#2ec4b6",
               "In-processing": "#7b9cda", "In-processing Algorithms": "#7b9cda",
               "Post-processing": "#f4845f", "Postprocessing Algorithms": "#f4845f",
               "Sequential Algorithms": "#d966a0", "Sequential Models": "#d966a0"}


def val(lib, name, col):
    r = summary[(summary.library == lib) & (summary.method == name) & (summary.metric == col)]
    return (r["mean"].iloc[0], r["ci_low"].iloc[0], r["ci_high"].iloc[0]) if len(r) else (np.nan,) * 3


# (1) Subgroup sensitivity: TPR of each ethnicity group, per configuration, with 95% CIs
ETH_SHOW = [(0, "Asian", "#2ec4b6", "o"), (1, "Black", "#f4845f", "s"), (2, "Hispanic", "#7b9cda", "^"), (4, "White", "#d966a0", "D")]
fig, axes = plt.subplots(2, 1, figsize=(15, 10))
for ax, lib in zip(axes, ["Fairlearn", "AIF360"]):
    names = [n for n, _, _ in METHODS[lib] if (lib, n) in BOOT]
    x = np.arange(len(names))
    for j, (c, nm, colr, mk) in enumerate(ETH_SHOW):
        v = np.array([val(lib, n, f"TPR|{nm}") for n in names]) * 100
        xx = x + (j - 1.5) * 0.13
        ax.errorbar(xx, v[:, 0], yerr=np.clip(np.array([v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]], dtype=float), 0, None), fmt=mk, color=colr, ms=5, capsize=2, lw=1, label=nm)
        for xi, n, yi in zip(xx, names, v[:, 0]):
            r = summary[(summary.library == lib) & (summary.method == n) & (summary.metric == f"TPR|{nm}")]
            tag = {"decrease": "↓", "increase": "↑"}.get(r["significant_change"].iloc[0], "") if len(r) else ""
            if tag:
                ax.text(xi, yi + 3, tag, ha="center", fontsize=9, color=colr, fontweight="bold")
    o = np.array([val(lib, n, "Sensitivity") for n in names]) * 100
    ax.plot(x, o[:, 0], "k_", ms=22, mew=2, label="All patients")
    ax.set_xticks(x); ax.set_xticklabels([SHORT.get(n, n) for n in names], fontsize=9)
    ax.set_ylim(0, 105); ax.set_ylabel("Sensitivity (TPR, %)"); ax.grid(axis="y", ls="--", alpha=0.4)
    ax.set_title(f"{lib}: COVID-19 sensitivity by ethnicity (mean of {len(SEEDS)} seeds, 95% CI; "
                 "↓/↑ significant change vs. original, Holm)", fontsize=11)
axes[0].legend(ncol=5, fontsize=9, loc="lower left")
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Subgroup-Sensitivity.png"), dpi=150, bbox_inches="tight"); plt.show()

# (2) Fairness-utility trade-off: balanced accuracy against the joint sex x ethnicity disparity
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, lib, met, lab in [(axes[0], "Fairlearn", "Gender+Ethnicity|EOD", "Equalized odds difference, sex x ethnicity (%)"),
                          (axes[1], "AIF360", "Gender+Ethnicity|AOD", "Average odds difference, sex x ethnicity (%)")]:
    for n, _, st in METHODS[lib]:
        if (lib, n) not in BOOT:
            continue
        fx, fl, fh = np.array(val(lib, n, met)) * 100
        uy, ul, uh = np.array(val(lib, n, "Balanced accuracy")) * 100
        ax.errorbar(fx, uy, xerr=np.clip(np.array([[fx - fl], [fh - fx]], dtype=float), 0, None),
                    yerr=np.clip(np.array([[uy - ul], [uh - uy]], dtype=float), 0, None), fmt="o", ms=7,
                    color=STAGE_COLOR.get(st, "#333"), capsize=2, lw=1)
        ax.annotate(SHORT.get(n, n), (fx, uy), textcoords="offset points", xytext=(5, 5), fontsize=8.5)
    ax.set_xlabel(lab); ax.set_ylabel("Balanced accuracy (%)"); ax.grid(ls="--", alpha=0.4); ax.set_title(lib)
handles = [mpatches.Patch(color=c, label=l) for l, c in [("Original", "#333333"), ("Pre-processing", "#2ec4b6"),
           ("In-processing", "#7b9cda"), ("Post-processing", "#f4845f"), ("Sequential", "#d966a0")]]
axes[1].legend(handles=handles, fontsize=9, loc="best")
fig.suptitle(f"Fairness-utility trade-off (mean of {len(SEEDS)} seeds, 95% CI; lower left of each panel = fairer, upper = more useful)", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Fairness-Utility.png"), dpi=150, bbox_inches="tight"); plt.show()

# (3) Ablations: AUC, balanced accuracy and disparity of the full model and its input ablations
names = [n for n, _, _ in METHODS["Ablation"] if ("Ablation", n) in BOOT]
panels = [("AUC", "AUC (%)"), ("Balanced accuracy", "Balanced accuracy (%)"),
          ("Ethnicity|EOD", "Equalized odds difference, ethnicity (%)"), ("Gender|EOD", "Equalized odds difference, sex (%)")]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (met, lab) in zip(axes, panels):
    v = np.array([val("Ablation", n, met) for n in names]) * 100
    cols = ["#333333", "#7b9cda", "#2ec4b6", "#8bc34a", "#f4845f"][:len(names)]
    ax.bar(np.arange(len(names)), v[:, 0], color=cols, edgecolor="black", yerr=np.clip(np.array([v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]], dtype=float), 0, None), capsize=3)
    for xi, n, yi, hi in zip(np.arange(len(names)), names, v[:, 0], v[:, 2]):
        r = summary[(summary.library == "Ablation") & (summary.method == n) & (summary.metric == met)]
        tag = {"decrease": "↓", "increase": "↑"}.get(r["significant_change"].iloc[0], "") if len(r) else ""
        ax.text(xi, hi + 1, f"{yi:.1f}{tag}", ha="center", fontsize=8.5)
    ax.set_xticks(np.arange(len(names))); ax.set_xticklabels([textwrap.fill(n, 10) for n in names], fontsize=8.5)
    ax.set_title(lab, fontsize=10); ax.grid(axis="y", ls="--", alpha=0.4)
    if met in ("AUC", "Balanced accuracy"):
        ax.set_ylim(40, 100)
fig.suptitle(f"Input ablations of the Original Model (mean of {len(SEEDS)} seeds, 95% CI; ↓/↑ significant change vs. full model, Holm)", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Ablation.png"), dpi=150, bbox_inches="tight"); plt.show()
print(sorted(os.listdir(PLOT_DIR)))
